# Radio Bulletin Generator — Three Languages

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Generate natural, radio-ready weather bulletins in **three languages**:
1. **English** (Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

- **Target length**: ~300 words (~2 minutes at 150 wpm radio reading pace)
- **Style**: Philippine radio broadcast — authoritative, clear, flowing prose
- **Model**: Gemma 4 E4B via Ollama (text-only — no vision needed here)
- **Input**: Full markdown bulletin text (from notebook 04)
- **Output**: Plain text scripts saved to `data/radio_bulletins/`

### Two sample bulletins
1. `PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito
2. `PAGASA_22-TC02_Basyang_TCA#01` — Tropical Cyclone Alert, Basyang

**Total output**: 6 scripts (2 bulletins × 3 languages)

In [1]:
import json
import requests
import time
from pathlib import Path

OLLAMA_API = "http://localhost:11434"
MODEL_NAME = "gemma4:e4b"
TIMEOUT = 10 * 60  # 10 minutes

data_dir = Path("../data")
markdown_dir = data_dir / "gemma4_results"
output_dir = data_dir / "radio_bulletins"
output_dir.mkdir(exist_ok=True)

# The two bulletins we are working with
SAMPLES = [
    "PAGASA_20-19W_Pepito_SWB#01",
    "PAGASA_22-TC02_Basyang_TCA#01",
]

# Verify Ollama is running
try:
    r = requests.get(f"{OLLAMA_API}/api/tags", timeout=5)
    names = [m['name'] for m in r.json()['models']]
    status = "\u2713" if any(MODEL_NAME in n for n in names) else "\u26a0\ufe0f  model not found"
    print(f"\u2713 Ollama running \u2014 {MODEL_NAME}: {status}")
except Exception as e:
    print(f"\u2717 Ollama not reachable: {e}")

print(f"\u2713 Output dir: {output_dir.absolute()}")


✓ Ollama running — gemma4:e4b: ✓
✓ Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins


## 1. Load Bulletin Markdown

Load the raw markdown text extracted by notebook 04. This is the primary input — no structured JSON used.

In [2]:
def load_bulletin(stem: str) -> dict:
    """Load the raw markdown for a bulletin stem (primary input for generation)."""
    markdown_path = markdown_dir / f"{stem}_markdown.md"

    if not markdown_path.exists():
        raise FileNotFoundError(f"Markdown file not found: {markdown_path}")

    markdown = markdown_path.read_text(encoding="utf-8")

    return {"stem": stem, "markdown": markdown}


bulletins = [load_bulletin(s) for s in SAMPLES]

for b in bulletins:
    print(f"\n{b['stem']}")
    print(f"  Markdown: {len(b['markdown'])} chars")
    print(f"  Preview:  {b['markdown'][:120].strip()!r}")



PAGASA_20-19W_Pepito_SWB#01
  Markdown: 4825 chars
  Preview:  '## REPUBLIC OF THE PHILIPPINES\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PHILIPPINE ASTROPHYSICAL, GEOPHYSICAL AND ASTR'

PAGASA_22-TC02_Basyang_TCA#01
  Markdown: 3224 chars
  Preview:  '# PHILIPPINE ARCHIPELAGO\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PAGASA**\nPhilippine Atmospheric, Geophysical and Ast'


## 2. Radio Bulletin Prompts — Three Languages

Generate radio bulletins in:
1. **English** (standard Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

All versions maintain the same style:
- Flowing prose — no bullet points, no tables, no markdown

- ~300 words (~2 minutes reading time)- Clear structure: opening → current situation → forecast → affected areas → public safety → closing

- Repeat storm name and signal areas (radio listeners may tune in mid-broadcast)- Plain spoken numbers

In [ ]:
PROMPTS = {
    "en": {
        "system": """You are converting a PAGASA typhoon bulletin into a plain spoken weather announcement in English.

STYLE:
- Write as if explaining to a neighbour — conversational, simple, direct
- No broadcaster language, no formal sign-offs, no station IDs, no "Good morning listeners"
- Short sentences. Common words. Anyone should understand this.
- Use digits for numbers (e.g. "25 kilometres per hour", "Signal 2")
- Cover: what the storm is, where it is, where it is going, who is affected, what people should do
- DO NOT add information that is not in the original bulletin

PHILIPPINE PLACE NAME PRONUNCIATION — spell these phonetically so they are read correctly by the TTS engine:
  - Catanduanes → ka-tan-du-a-nes
  - Cagayan → ka-ga-yan
  - Isabela → i-sa-be-la
  - Visayas → bi-sa-yas
  - Mindanao → min-da-naw
  - Quezon → ke-son
  - Batanes → ba-ta-nes
  - Samar → sa-mar
  - Leyte → ley-te
  - Palawan → pa-la-wan
  - Bicol → bi-kol
  - Masbate → mas-ba-te
  - Surigao → su-ri-ga-o
  - Pangasinan → pa-nga-si-nan
  - Zamboanga → zam-bo-an-ga

FORMATTING: Plain flowing prose only. No headings, no bullet points, no bold, no markdown. Paragraph breaks (blank lines) between ideas.

LENGTH: About 300 words.""",

        "user_template": (
            "Convert this PAGASA weather bulletin into a plain conversational English announcement.\n\n"
            "{markdown}\n\n"
            "Write the announcement now. Conversational tone, simple words, about 300 words. "
            "No headings, no bullet points, no markdown. Spell Filipino place names phonetically."
        ),
    },

    "tl": {
        "system": """Ikaw ay nagsusulat ng simpleng pahayag tungkol sa isang malakas na bagyo para marinig ng mga tao.

ESTILO — PURO TAGALOG, WALANG INGLES:
- Magsulat na parang nagkukwento ka sa isang kapitbahay — simple, natural, walang paligoy-ligoy
- BAWAL ang mga salitang Ingles maliban sa mga pangalan ng tao at lugar (hal. Pepito, Catanduanes, Luzon)
- Gamitin ang pang-araw-araw na Tagalog — hindi pormal, hindi opisyal, hindi balita sa TV
- Kung may teknikal na salita, i-spell phonetically gamit ang gitling:
    tai-pun, tro-pi-kal di-pre-syon, sig-nal nam-ber wan, ki-lo-me-tro ba-wat o-ras,
    storm serj, land-pol, kos-tal, pag-asa, pore-kast, wor-ning, ad-bay-so-ri
- Maikling pangungusap. Madaling salita. Dapat maintindihan ng sinumang Pilipino.
- Gamitin ang mga digit para sa mga numero (hal. 25 kilometro bawat oras, Signal 2)
- Sabihin: ano ang bagyo, nasaan ito, saan patungo, sino ang maaapektuhan, ano ang dapat gawin
- HUWAG magdagdag ng impormasyon na wala sa orihinal na bulletin

FORMATTING: Natural na daloy ng prosa. Walang headings, walang bullets, walang bold, walang markdown. Blank lines sa pagitan ng mga talata.

HABA: Mga 300 salita.""",

        "user_template": (
            "I-convert ang PAGASA weather bulletin na ito sa simpleng pahayag sa Tagalog.\n\n"
            "{markdown}\n\n"
            "Isulat ang pahayag ngayon. Puro Tagalog — walang Ingles maliban sa mga pangalan. "
            "Natural na tono, madaling salita, mga 300 salita. Walang headings, walang markdown."
        ),
    },

    "ceb": {
        "system": """Ikaw nagsulat og simple nga pahimangno bahin sa usa ka kusog nga bagyo para madungog sa mga tawo.

ESTILO — PURO CEBUANO, WALAY ENGLISH:
- Pagsulat sama sa imong gisulti sa imong silingan — simple, natural, dili komplikado
- BAWAL ang mga pulong nga Ingles gawas sa mga pangalan sa tawo ug lugar (hal. Pepito, Catanduanes, Luzon)
- Gamita ang inadlaw-adlaw nga Cebuano — dili pormal, dili opisyal, dili balita sa TV
- Kung adunay teknikal nga pulong, i-spell phonetically gamit ang gitling:
    tai-pun, tro-pi-kal di-pre-syon, sig-nal nam-ber wan, ki-lo-me-tros sa usa ka oras,
    storm serj, land-pol, kos-tal, pag-asa, pore-kast, wor-ning, ad-bay-so-ri
- Mubo nga mga sentence. Sayon nga mga pulong. Kinahanglan masabtan sa tanan nga Pilipino.
- Gamita ang mga digit para sa mga numero (hal. 25 kilometros sa usa ka oras, Signal 2)
- Isulti: unsa ang bagyo, asa kini, asa padulong, kinsa ang maapektuhan, unsa ang buhaton
- AYAW pagdugang og impormasyon nga wala sa orihinal nga bulletin

FORMATTING: Natural nga daloy sa prosa. Walay headings, walay bullets, walay bold, walay markdown. Blank lines tali sa mga paragraph.

GITAS-ON: Mga 300 ka pulong.""",

        "user_template": (
            "I-convert ang PAGASA weather bulletin nga kini ngadto sa simple nga pahimangno sa Cebuano.\n\n"
            "{markdown}\n\n"
            "Isulat ang pahimangno karon. Puro Cebuano — walay English gawas sa mga pangalan. "
            "Natural nga tono, sayon nga mga pulong, mga 300 ka pulong. Walay headings, walay markdown."
        ),
    },
}


def build_user_prompt(bulletin: dict, language: str) -> str:
    """Build the user prompt using the full markdown bulletin text as input."""
    template = PROMPTS[language]["user_template"]
    return template.format(markdown=bulletin["markdown"])


print("\u2713 Prompts defined for 3 languages")
for lang, prompts in PROMPTS.items():
    print(f"  {lang}: system={len(prompts['system'])} chars, template={len(prompts['user_template'])} chars")


## 3. Generate Radio Bulletins — All Three Languages

Call Gemma 4 for each bulletin in English, Tagalog, and Cebuano (6 total scripts).

In [4]:
def generate_radio_bulletin(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a radio broadcast script for one bulletin in the specified language."""
    stem = bulletin["stem"]
    lang_names = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}
    print(f"\nGenerating: {stem} ({lang_names[language]})")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": PROMPTS[language]["system"]},
            {"role": "user", "content": build_user_prompt(bulletin, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    elapsed = time.time() - t_start

    script = response.json().get("message", {}).get("content", "").strip()

    # Word count (strip markdown syntax for accurate spoken-word estimate)
    import re
    plain = re.sub(r"[#*_`>\-]", "", script)
    word_count = len(plain.split())
    reading_minutes = word_count / 150  # ~150 wpm for radio

    # Save as markdown file
    out_path = output_dir / f"{stem}_radio_{language}.md"
    out_path.write_text(script, encoding="utf-8")

    print(f"  \u2713 Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}  |  Est. reading time: {reading_minutes:.1f} min")
    print(f"  Saved \u2192 {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "script": script,
        "word_count": word_count,
        "reading_minutes": reading_minutes,
        "elapsed": elapsed,
    }


results = []
for bulletin in bulletins:
    for lang in ["en", "tl", "ceb"]:
        result = generate_radio_bulletin(bulletin, lang)
        results.append(result)

print(f"\n\u2713 Done \u2014 {len(results)} scripts generated ({len(bulletins)} bulletins \u00d7 3 languages)")



Generating: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 26.9s
  Words: 304  |  Est. reading time: 2.0 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_en.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 35.6s
  Words: 385  |  Est. reading time: 2.6 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_tl.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 33.5s
  Words: 375  |  Est. reading time: 2.5 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_ceb.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 17.9s
  Words: 328  |  Est. reading time: 2.2 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_en.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 37.6s
  Words: 371  |  Est. reading time: 2.5 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_tl.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 32.8s
  Words: 343  |  Est. reading time: 2.3 min
  Saved → PAGASA_22-TC02_Basyan

## 6. TTS Plain Text Generation — All Three Languages

Generate TTS-optimized plain text versions of all radio scripts via a second
Gemma4 prompt. These files feed directly into notebook 08 — no markdown stripping needed.

**Output:** `data/radio_bulletins/{stem}_tts_{lang}.txt` (CEB, TL, EN)

In [5]:
TTS_PROMPTS = {
    "ceb": {
        "system": """Ikaw usa ka espesyalista sa Cebuano nga nagsulat og plain text nga angay para sa text-to-speech synthesis.

Ang imong trabaho:
- Basaha ang markdown radio script nga gihatag
- Isulat kini pag-usab isip natural nga flowing prose SA CEBUANO LAMANG — walay markdown
- WALA markdown: wala headings (#), wala bullet points (-), wala asterisks (*), wala bold/italic
- Para sa mga English proper nouns o teknikal nga termino, i-spell sila phonetically sa Cebuano:
  - PAGASA → pa-ga-sa
  - Northern Luzon → nor-dern lu-son
  - Signal Number One / Two / Three → sig-nal nam-ber wan / tu / tri
  - tropical depression → tro-pi-kal di-pre-syon
  - tropical storm → tro-pi-kal storm
  - kilometers per hour → ki-lo-me-tros sa usa ka oras
  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west
- Pahimusa ang paragraph structure: blank lines tali sa mga paragraph
- AYAW pagdugang og bisan unsa nga texto nga wala sa orihinal nga script
- Output: plain text lamang, walay bisan unsang markup o formatting characters""",

        "user_template": (
            "Basaha kining markdown radio script ug isulat kini pag-usab isip TTS-ready plain Cebuano text.\n\n"
            "{markdown}\n\n"
            "Isulat ang plain Cebuano text karon. Cebuano nga pulong lamang, phonetically spelled kung "
            "kinahanglan, paragraph breaks (blank lines) para sa natural nga pausing. Walay markdown."
        ),
    },
    "tl": {
        "system": """Ikaw ay isang espesyalista sa Tagalog na sumusulat ng plain text na angkop para sa text-to-speech synthesis.

Ang iyong trabaho:
- Basahin ang markdown radio script na ibinigay
- Isulat muli ito bilang natural na flowing prose SA TAGALOG LAMANG — walang markdown
- WALANG markdown: walang headings (#), walang bullet points (-), walang asterisks (*), walang bold/italic
- Para sa mga English proper nouns o teknikal na termino, i-spell ang mga ito nang phonetically sa Tagalog:
  - PAGASA → pa-ga-sa
  - Northern Luzon → nor-dern lu-son
  - Signal Number One / Two / Three → sig-nal nam-ber wan / tu / tri
  - tropical depression → tro-pi-kal di-pre-syon
  - tropical storm → tro-pi-kal storm
  - kilometers per hour → ki-lo-me-tro ba-wat o-ras
  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west
- Panatilihin ang paragraph structure: blank lines sa pagitan ng mga paragraph
- HUWAG magdagdag ng anumang texto na wala sa orihinal na script
- Output: plain text lamang, walang anumang markup o formatting characters""",

        "user_template": (
            "Basahin ang markdown radio script na ito at isulat muli ito bilang TTS-ready plain Tagalog text.\n\n"
            "{markdown}\n\n"
            "Isulat ang plain Tagalog text ngayon. Tagalog na salita lamang, phonetically spelled kung "
            "kinakailangan, paragraph breaks (blank lines) para sa natural na pausing. Walang markdown."
        ),
    },
    "en": {
        "system": """You are a specialist in writing plain text suitable for text-to-speech synthesis.

Your task:
- Read the provided markdown radio script
- Rewrite it as natural flowing prose IN ENGLISH ONLY — no markdown
- NO markdown: no headings (#), no bullet points (-), no asterisks (*), no bold/italic
- Keep technical terms clear and pronounceable
- Maintain paragraph structure: blank lines between paragraphs
- DO NOT add any text that wasn't in the original script
- Output: plain text only, no markup or formatting characters""",

        "user_template": (
            "Read this markdown radio script and rewrite it as TTS-ready plain English text.\n\n"
            "{markdown}\n\n"
            "Write the plain English text now. Clear pronunciation-friendly English, "
            "paragraph breaks (blank lines) for natural pausing. No markdown."
        ),
    },
}


def build_tts_prompt(markdown_script: str, language: str) -> str:
    """Build the user prompt for TTS text generation."""
    return TTS_PROMPTS[language]["user_template"].format(markdown=markdown_script)


print("✓ TTS_PROMPTS defined for CEB, TL, EN")
print(f"  ceb: system={len(TTS_PROMPTS['ceb']['system'])} chars")
print(f"  tl:  system={len(TTS_PROMPTS['tl']['system'])} chars")
print(f"  en:  system={len(TTS_PROMPTS['en']['system'])} chars")

✓ TTS_PROMPTS defined for CEB, TL, EN
  ceb: system=1040 chars
  tl:  system=1055 chars
  en:  system=519 chars


In [6]:
def generate_tts_text(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a TTS-ready plain text script for one bulletin."""
    stem = bulletin["stem"]
    lang_names = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}
    print(f"\nGenerating TTS text: {stem} ({lang_names[language]})")

    markdown_script = (output_dir / f"{stem}_radio_{language}.md").read_text(encoding="utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": TTS_PROMPTS[language]["system"]},
            {"role": "user", "content": build_tts_prompt(markdown_script, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    elapsed = time.time() - t_start

    tts_text = response.json().get("message", {}).get("content", "").strip()

    out_path = output_dir / f"{stem}_tts_{language}.txt"
    out_path.write_text(tts_text, encoding="utf-8")

    word_count = len(tts_text.split())
    print(f"  ✓ Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}")
    print(f"  Saved → {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "tts_text": tts_text,
        "word_count": word_count,
        "elapsed": elapsed,
    }


# Generate TTS text for both bulletins — all three languages
tts_results = []
for bulletin in bulletins:
    for lang in ["ceb", "tl", "en"]:
        result = generate_tts_text(bulletin, lang)
        tts_results.append(result)

print(f"\n✓ Done — {len(tts_results)} TTS text files generated")

# Preview first bulletin (CEB)
print("\nPREVIEW — CEB TTS text (first 500 chars)")
print("=" * 60)
ceb_result = [r for r in tts_results if r["language"] == "ceb"][0]
print(ceb_result["tts_text"][:500])
print("...")


Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 50.9s
  Words: 337
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_ceb.txt

Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 45.2s
  Words: 349
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_tl.txt

Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 8.6s
  Words: 286
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_en.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 46.4s
  Words: 304
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_ceb.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 27.5s
  Words: 344
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_tl.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 25.3s
  Words: 315
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_en.txt

✓ Done — 6 TTS text files generated

PREVIEW — CEB TTS text (first 500 chars)
Maayong buntag, mga kauban! Kini inyon

## 4. Review Output

Display each generated script for manual inspection (grouped by bulletin, showing all languages).

In [7]:
from IPython.display import display, Markdown
from collections import defaultdict

grouped = defaultdict(list)
for r in results:
    grouped[r["stem"]].append(r)

lang_labels = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}

for stem, versions in grouped.items():
    display(Markdown(f"---\n# {stem}\n---"))
    for version in versions:
        lang = version["language"]
        meta = (
            f"**Language:** {lang_labels[lang]}  \u2502  "
            f"**Words:** {version['word_count']}  \u2502  "
            f"**Est. reading time:** {version['reading_minutes']:.1f} min"
        )
        display(Markdown(f"## {lang_labels[lang]} Version\n\n{meta}\n\n---\n\n{version['script']}"))


---
# PAGASA_20-19W_Pepito_SWB#01
---

## English Version

**Language:** English  │  **Words:** 304  │  **Est. reading time:** 2.0 min

---

# PAGASA Weather Advisory on Tropical Depression PEPITO

## Current Situation
Good morning, Philippines. This is a weather advisory update from the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA.

At this time, the area east of Catanduanes has intensified into **Tropical Depression "PEPITO."** This is the first official update regarding **Tropical Depression "PEPITO"** for today, October nineteenth. While there is no tropical cyclone wind signal currently in effect, we are monitoring the storm’s intensity closely. PAGASA advises the public to remain highly vigilant.

## Forecast Track
Moving on to the forecast, **Tropical Depression "PEPITO"** is expected to follow a west-northwest to westward track, heading toward Northern Luzon. The storm has the potential to intensify further into a tropical storm or even a typhoon. The most critical forecast is that **Tropical Depression "PEPITO"** is predicted to make landfall over the eastern coast of Northern Luzon on the afternoon of October twenty-first.

Residents in Catanduanes, Northern Samar, and Samar are expected to be the most severely affected areas.

## Affected Areas & Public Safety Advisory
Coastal waters throughout the Visayas and Northern Luzon are expected to experience moderate to strong seas, potentially reaching between two point five and four point five meters.

Residents are urged to prepare for intense rainfall and strong winds. Specifically, areas from the Bicol Region, through Catanduanes, all the way to Northern Luzon, should be on high alert. The primary hazards to expect include potential flooding and rough waters.

## Closing
We reiterate the warning that while **Tropical Depression "PEPITO"** is currently moving, the threat of tropical conditions is high. The public is strongly advised to take appropriate measures, secure loose belongings, and stay tuned to official PAGASA announcements.

We will provide the next Severe Weather Bulletin update at eleven o’clock in the morning today. Stay safe, and let’s continue to monitor the latest advisories.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 385  │  **Est. reading time:** 2.6 min

---

# TROPICAL DEPRESSION "PEPITO" SEVERE WEATHER BULLETIN: UPDATE

*(Sound of gentle, serious background music fades slightly)*

**Broadcast Meteorologist:** Magandang umaga po, Pilipino. Ito po ang inyong weather bulletin mula sa **PAGASA**, o ang Philippine Atmospheric, Geophysical and Astronomical Services Administration. Patuloy po tayong magbabantay sa mga pagbabago sa ating kalikasan.

### Kasalukuyang Sitwasyon

Sa ngayon, makakaranas tayo ng pag-uugnay sa low pressure area na nag-develop na at itinalagang **Tropical Depression "Pepito"**. Ang **Pepito** ay kasalukuyang nasa silangan ng Catanduanes. Patuloy po itong inaasahang gagalaw nang pa-kanluran hanggang pa-hilagang-kanluran. Mahalagang tandaan na may posibilidad na palakasin pa ang **Pepito** at maging Tropical Storm o posibleng Typhoon sa paglipas ng panahon.

### Forecast Track

Patungong forecast, inaasahang aabot sa baybayin ng Hilagang Luzon ang **Pepito** sa hapon ng ika-dalawampu’ing-isa ng Oktubre. Ang mga mangangalaga po sa kaligtasan ay dapat maging handa sa pag-uugnay nito sa rehiyon. Patuloy po nating babantayan ang paggalaw ng bagyong ito.

### Mga Apektadong Lugar

Ang mga lugar na inaasahang magiging pinaka-apektado ay ang Catanduanes, Northern Samar, Samar, at lalo na ang mga kabayanan sa Hilagang Luzon. Habang naglalakbay ang bagyo, maghanda po sa pagbuhos ng malakas na pag-ulan at may kasamang malakas na hangin.

Para sa ating mga nasa baybayin, inaasahan ang pagtaas ng alon na umaabot sa dalawang-kuwarto hanggang apat at kalahating metro. Ito ay isang malaking babala para sa mga gumagamit ng bangka at maliliit na sasakyang pang-dagat.

### Payo sa Kaligtasan

Hinihikayat po ang lahat ng residente na maging kalmado ngunit handa. Dahil sa pagiging malakas ng posibleng epekto ng **Pepito**, inirerekomenda na maghanda na ng mga emergency kits. Kung kayo ay residente sa mga baybayin at low-lying areas, makinig po sa mga paalala ng inyong lokal na pamahalaan, at maging handa sa paglikas kung ito ay utusan.

Ang pangunahing babala po ay patungkol sa matinding pag-ulan at malakas na hangin. Huwag po tayong maging kampante.

### Pangwakas

Ang kasalukuyang signal po ay wala pang Tropical Cyclone Wind Signal. Ngunit, muli po, tandaan natin ang pangangailangan na maging handa sa mga babala ng **Tropical Depression "Pepito"**.

Muli po ninyong tandaan: *Maghanda sa matinding pag-ulan at malakas na hangin.*

Para sa susunod na update, kami po ay babalik sa oras na alas-onse ng umaga. Manatiling kalmado, at patuloy po ninyong pinapakinggan ang inyong mga lokal na weather advisories.

*(Background music gently rises and then fades out)*

## Cebuano Version

**Language:** Cebuano  │  **Words:** 375  │  **Est. reading time:** 2.5 min

---

# PAGASA Weather Update: Tropical Depression "Pepito" - Alert Bulletin

*(Sound of light, upbeat jingle fades out)*

**Broadcast Station Voice:** "Good morning, mga kauban! Kini inyong paminaw sa [Name of Radio Station]. Ato man usab atong i-check ang mga kondisyon sa panahon pinaagi sa mga pagpahibalo gikan sa PAGASA, ang Philippine Atmospheric, Geophysical and Astronomical Services Administration.

Karon, mga higala, usa ka importante kaayo nga mensahe ang atoang ipaambit mahitungod sa Tropical Depression nga gitawag og **Pepito**. Ato pang hisgutan ang kalihukan ni **Pepito**.

## Unsay Nahitabo Karon

Ang Tropical Depression **Pepito** nag-develop og kusog sa rehiyon sa Silangang Katanduanes. Ang PAGASA nagtago nga kini adunay posibilidad nga mo-intensify pa, ug basin mo-abot pa kini og Tropical Storm, o bisan pa nga Typhoon.

Apan, pag-amping, karon nga wala pa kitay signal nga nag-epektibo sa dagat. Pero, kinahanglan kitang andam.

## Asa Padulong ang Bagyo

Unya kini, mga kauban, ang atong ituki nga track ni **Pepito**. Ang prediksyon nag-ingon nga si **Pepito** mo-west-northwestward, ug ang target niini mao ang pag-abot sa baybayon sa Northern Luzon, posible kini sa hapon sa Oktubre ika-21.

Kinsang mga lugar ang pinaka-apektado? Ang mga kritikal nga rehiyon mao ang Catanduanes, Northern Samar, Samar, Southern Leyte, Bicol Region, ug labaw sa tanan, ang mga coastal areas sa Northern Luzon.

## Hazard ug Pag-andam

Mga kauban, ang labing importante nga ipahinumdoman ninyo karon mao ang *andam* sa mga kalisdanan.

Base sa pagtagad, ang mga rehiyon nga kining mga ngalan, mag-atubang og kusog nga pag-ulan ug kusog nga hangin.

*   **Para sa mga mandaragat:** Pasidaan ko mo, ang pag-atubang sa dagat sa mga bahin nga kini nga mga ngalan, ang mga dagat mahimong maoy *moderate to strong*. Busa, palihug, ayaw og kaayog kalayo sa dagat, ilabi na kon gamay ra kaayo mo nga sakay.
*   **Importante nga:** Kahitam-an ninyo ang mga pag-andam sa inyong komunidad—kana ang pag-andam sa pag-ulan, pag-andam sa uban, ug pagbantay sa mga pagpahibalo gikan sa inyong lokal nga government.

## Pangwakas

Karon, mga higala, ang pag-andam maoy atong pinakainiton nga kauban. Paghinam-i og mga pagkaon, pagpabilin sa mga emergency kit, ug makigkomunikaray sa inyong mga pamilya.

Ang atong mo-monitor pag-ayo ni **Pepito**. Padayon kitang mag-uban sa PAGASA.

Unya ha, mga kauban. Magkita ta usab sa atong sunod nga weather update. Kabay pa kita!

*(Sound of jingle fades up)*

---
# PAGASA_22-TC02_Basyang_TCA#01
---

## English Version

**Language:** English  │  **Words:** 328  │  **Est. reading time:** 2.2 min

---

# Tropical Storm "Malakas" Advisory Bulletin

## Current Situation
Good day, listeners. We bring you this weather update from the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA.

At this time, **Tropical Storm "Malakas"** is located approximately two point zero zero kilometers east of Mindanao, maintaining a maximum sustained wind of seventy-five kilometers per hour, with gustiness reaching up to ninety kilometers per hour. **Tropical Storm "Malakas"** is moving toward the west-northwest at a speed of fifteen kilometers per hour. Importantly, **Tropical Storm "Malakas"** remains outside the Philippine Area of Responsibility.

## Forecast Track
Moving on to the forecast track, our latest data shows that the storm is projected to continue strengthening and moving generally westward. According to PAGASA's 72-hour forecast, the system is expected to reach Typhoon intensity—or **Signal Number Ten**—as it passes over the central seas.

We want the public to be aware that even as the storm intensifies, its path remains far offshore, easing concern regarding immediate landfall.

## Affected Areas
To clarify, PAGASA advises that **Tropical Storm "Malakas"** is expected to remain outside the Philippine Area of Responsibility throughout its projected path. This means residents across Luzon, Visayas, and Mindanao are not under a tropical cyclone warning issued by PAGASA at this time.

However, all regions are still advised to prepare for potential impacts from strong winds, rough seas, and potential rain, as the weather system is still highly active in the wider region.

## Public Safety Advisory
Residents are urged to remain vigilant. While the immediate threat is offshore, we strongly advise communities to review their disaster preparedness plans, secure loose outdoor items, and monitor official PAGASA channels. Remember, should the storm approach, the most critical warning signal to be prepared for is the designation of a Typhoon, requiring immediate adherence to local government unit protocols.

## Closing
This has been your weather update. We will provide the next full advisory in two hours, at eleven o'clock in the morning. Stay safe, and we thank you for listening.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 371  │  **Est. reading time:** 2.5 min

---

# PAGASA Weather Advisory: Tropical Storm "MALAKAS" Update

## Kasalukuyang Sitwasyon

Mula sa Philippine Atmospheric, Geophysical and Astronomical Services Administration, o PAGASA, narito po ang inyong weather advisory para sa **Tropical Storm "MALAKAS"**.

Sa kasalukuyan, tinatantya po na ang sentro ng **Tropical Storm "MALAKAS"** ay matatagpuan sa labas po ng Philippine Area of Responsibility, o PAR. Sa oras na ito, ang bilis ng hangin ay tinatayang pitumpu't limang kilometro bawat oras, na may pagbugso na umaabot sa siyamnapu't kilometro bawat oras. Ang patuloy na pagsubaybay ay mahalaga dahil patuloy itong umaandar sa direksyon na west-northwest.

## Forecast Track

Patungo sa forecast, ipinapakita po na ang **Tropical Storm "MALAKAS"** ay patuloy na lalayo sa ating mga kabayanan. Inaasahan po na ang bagyo ay magpapatuloy sa paghakbang nang westward-northwestward, at mananatili pa rin po itong nasa labas ng ating PAR.

Gayunpaman, habang naglalayag ito sa karagatan, pinag-aaralan pa rin po ng mga meteorologo ang potensyal na paglakas o paghina nito. Kinakailangan po nating tandaan na bagama't nasa labas po ng bansa ang bagyo, patuloy pa rin po nating binabantayan ang anumang posibleng pagbabago sa kondisyon ng hangin sa karagatan.

## Mga Apektadong Lugar at Payo sa Kaligtasan

Dahil ang **Tropical Storm "MALAKAS"** ay matatagpuan pa ring malayo sa ating mga rehiyon, sa ngayon ay walang specific signal na itinaas sa mga lalawigan.

Ngunit, hinihikayat po ang lahat ng residente na manatiling handa. Ipagpatuloy po natin ang pag-iingat laban sa mataas na alon, malakas na hangin, at anumang pagbaha na maaaring dulot ng mga bagyo sa nakalipas na araw.

Ang pinakamahalaga po ay ang pagpapanatili ng kaayusan at paghahanda. Patuloy po tayong mangangalap ng mga opisyal na anunsyo mula sa mga lokal na pamahalaan. Tandaan po natin, ang pangunahing pag-iingat ay **manatiling nakatutok sa mga babala ng PAGASA**.

## Pangwakas

Paulit-ulitin po namin, ang pag-iingat at pagpapanatili ng kaayusan ang ating pangunahing babala. Manatiling kalmado at laging nakatutok sa mga opisyal na anunsyo mula sa PAGASA.

Muli po ninyong tandaan: Kami po ay nagpapaalala na manatiling nakatutok sa mga babala ng PAGASA kaugnay ng pagdaan ng **Tropical Storm "MALAKAS"**.

Ito po ang inyong weather advisory mula sa PAGASA. Mag-uulat po kami ulit sa inyo sa susunod na iskedyul, sa alas-diyes ng umaga. Maraming salamat po at mag-ingat po kayo.

## Cebuano Version

**Language:** Cebuano  │  **Words:** 343  │  **Est. reading time:** 2.3 min

---

# Tropical Storm "MALAKAS" Update (Weather Advisory)

*(Sound Effect: Short, upbeat radio jingle fades out)*

**Broadcaster:** Maayong hapon, mga higala! Karon, atong mag-uban ning usa ka importanteng *weather update*. Kining atong mga impormasyon gikan ni PAGASA, ang Philippine Atmospheric, Geophysical and Astronomical Services Administration. Mao ni nga among kauban paghatag og mga update sa mga bagyo, mao nga pagtagad gyud mo.

***

## Unsay Nahitabo Karon

Karon ha, among atong hisgotan ang Tropical Storm **MALAKAS**. Sa paglabay sa kaadlawon, mga kauban, gipahibalo nga ang **MALAKAS** nagpabilin pa nga lig-on, pero importante nga bantayan nato ang iyang kadasig ug direksiyon.

Ang sentro sa **MALAKAS**, base sa ika-upat sa pagka-kaadlawon, nahimutang pa gihapon sa dagat, mga duha ka kilometro lamang east sa Mindanao. Ang iyang kusog nagpabilin nga mga pito ka lima ka kilometro kada oras ang hangin, naay kusog nga hangin nga moabot og kuwatro ka siyam ka kilometro kada oras.

***

## Asa Padulong ang Bagyo

Unya ha, kini ang labing importante nga hisgotan. Ang **MALAKAS** padulong sa West Northwest nga direksyon, mga kauban. Pero paminaw mo, kining bagyong **MALAKAS** *nagpabilin pa nga layo sa Philippine Area of Responsibility o PAR*.

Sa forecast nga mga adlaw, atong makita nga ang **MALAKAS** magpadayon sa paglayo pa gikan sa mga kalibutanon nga yuta sa atong Pilipinas. Magpadayon kini sa pag-west-northwest, mag-agi pa sa dagat sa pag-agi sa Visayas, ug magpadayon pa sa paglayo gikan sa pag-abot sa Luzon.

***

## Kinsa ang Apektado ug Unsa ang Kinahanglan Buhaton

Karon, nga ang **MALAKAS** nagpabilin nga layo sa atong teritoryo, walay dinalian nga peligro sa atong mga kabahin sa Pilipinas.

Apan, mga kauban, importante pa gihapon ang atong kahandaan. Kanunay nga bantayan ang mga opisyal nga pahibalo sa PAGASA ug sa inyong lokal nga mga awtoridad. Dili magpasakay sa mga rumor o unofficial reports.

***

## Pangwakas

So, sa pagpasiugda niining mga impormasyon, ang atong message mao: Bantayi pa gihapon ang **MALAKAS**, pero kalmado lang, mga kauban.

Mag-amping kamo, magpabilin nga updated, ug magkita-kita ta pag-usab sa among sunod nga weather update. Hangtod nianang higayon, mag-amping mo kanunay!

*(Sound Effect: Short, upbeat radio jingle fades in)*

## 5. Length Check

Verify word counts across all three languages are close to the 300-word target. If any are significantly short or long, we can adjust the prompts.

In [8]:
TARGET_WORDS = 300
WORDS_PER_MIN = 150

print("\nLENGTH SUMMARY")
print("=" * 60)
print(f"{'Bulletin':<35} {'Lang':>4} {'Words':>6}  {'Minutes':>7}  {'vs Target':>10}")
print("-" * 60)

for result in results:
    diff = result['word_count'] - TARGET_WORDS
    diff_str = f"+{diff}" if diff >= 0 else str(diff)
    label = result['stem'].replace('PAGASA_', '')
    lang = result['language'].upper()
    print(f"{label:<35} {lang:>4} {result['word_count']:>6}  {result['reading_minutes']:>6.1f}m  {diff_str:>10}")

print("-" * 60)
print(f"Target: {TARGET_WORDS} words ≈ {TARGET_WORDS/WORDS_PER_MIN:.0f} minutes at {WORDS_PER_MIN} wpm")
print(f"\n✓ {len(results)} scripts saved to: {output_dir.absolute()}")


LENGTH SUMMARY
Bulletin                            Lang  Words  Minutes   vs Target
------------------------------------------------------------
20-19W_Pepito_SWB#01                  EN    304     2.0m          +4
20-19W_Pepito_SWB#01                  TL    385     2.6m         +85
20-19W_Pepito_SWB#01                 CEB    375     2.5m         +75
22-TC02_Basyang_TCA#01                EN    328     2.2m         +28
22-TC02_Basyang_TCA#01                TL    371     2.5m         +71
22-TC02_Basyang_TCA#01               CEB    343     2.3m         +43
------------------------------------------------------------
Target: 300 words ≈ 2 minutes at 150 wpm

✓ 6 scripts saved to: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins
